<a href="https://colab.research.google.com/github/Lyna122/Data-analytic/blob/main/churn_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Customer Churn Analysis
## Real Data Analysis with Machine Learning

This notebook performs comprehensive churn analysis using a real telecom customer dataset.

In [1]:
# Install required packages
import sys
!{sys.executable} -m pip install pandas numpy matplotlib seaborn scikit-learn plotly kaleido --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 2.4 MB/s eta 0:00:00


In [16]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(" Libraries loaded")

 Libraries loaded


## 1. Load the Dataset

We'll use the Telco Customer Churn dataset from Kaggle.

In [14]:
# Download the dataset
import urllib.request

url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
urllib.request.urlretrieve(url, 'telco_churn.csv')

# Load data
df = pd.read_csv('telco_churn.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

Dataset shape: (7043, 21)

Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Data Exploration & Cleaning

In [19]:
# Basic info
print("DATASET INFORMATION")
df.info()


print("MISSING VALUES")
print("=" * 50)
print(df.isnull().sum())


print("STATISTICAL SUMMARY")
print("=" * 50)
df.describe()

DATASET INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [21]:
# Clean data
# Convert TotalCharges to numeric (some values are spaces)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Drop missing values (only 11 rows)
df = df.dropna()

# Convert Churn to binary
df['Churn_Binary'] = (df['Churn'] == 'Yes').astype(int)

print(f"Cleaned dataset shape: {df.shape}")
print(f"\nChurn Distribution:")
print(df['Churn'].value_counts())
print(f"\nChurn Rate: {df['Churn_Binary'].mean() * 100:.2f}%")

Cleaned dataset shape: (7032, 22)

Churn Distribution:
Churn
No     5163
Yes    1869
Name: count, dtype: int64

Churn Rate: 26.58%


## 3. CHURN RATE OVER TIME

In [22]:
# Create tenure groups
df['TenureGroup'] = pd.cut(df['tenure'],
                            bins=[0, 12, 24, 36, 48, 60, 72],
                            labels=['0-12', '12-24', '24-36', '36-48', '48-60', '60-72'])

# Calculate churn rate by tenure group
churn_by_tenure = df.groupby('TenureGroup')['Churn_Binary'].agg(['mean', 'count']).reset_index()
churn_by_tenure.columns = ['Tenure (months)', 'Churn Rate', 'Customer Count']
churn_by_tenure['Churn Rate'] = churn_by_tenure['Churn Rate'] * 100

# Create visualization
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(x=churn_by_tenure['Tenure (months)'],
           y=churn_by_tenure['Churn Rate'],
           name='Churn Rate (%)',
           marker_color='#ef4444'),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(x=churn_by_tenure['Tenure (months)'],
               y=churn_by_tenure['Customer Count'],
               name='Customer Count',
               line=dict(color='#3b82f6', width=3),
               mode='lines+markers'),
    secondary_y=True,
)

fig.update_xaxes(title_text="Customer Tenure")
fig.update_yaxes(title_text="Churn Rate (%)", secondary_y=False)
fig.update_yaxes(title_text="Number of Customers", secondary_y=True)

fig.update_layout(
    title='Churn Rate by Customer Tenure',
    height=500,
    hovermode='x unified'
)

fig.show()

print("\n Key Insight:")
print(f"Highest churn rate: {churn_by_tenure.iloc[0]['Churn Rate']:.1f}% in first 12 months")
print(f"Lowest churn rate: {churn_by_tenure.iloc[-1]['Churn Rate']:.1f}% after 60 months")


 Key Insight:
Highest churn rate: 47.7% in first 12 months
Lowest churn rate: 6.6% after 60 months


## 4. CUSTOMER SEGMENTATION BY CHURN

In [23]:
# Create customer segments
segments = ['Contract', 'InternetService', 'PaymentMethod', 'SeniorCitizen']

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Contract Type', 'Internet Service', 'Payment Method', 'Senior Citizen'),
    specs=[[{"type": "bar"}, {"type": "bar"}],
           [{"type": "bar"}, {"type": "bar"}]]
)

# Contract Type
contract_churn = df.groupby('Contract')['Churn_Binary'].mean() * 100
fig.add_trace(
    go.Bar(x=contract_churn.index, y=contract_churn.values,
           marker_color=['#10b981', '#f59e0b', '#ef4444'],
           showlegend=False),
    row=1, col=1
)

# Internet Service
internet_churn = df.groupby('InternetService')['Churn_Binary'].mean() * 100
fig.add_trace(
    go.Bar(x=internet_churn.index, y=internet_churn.values,
           marker_color=['#8b5cf6', '#ec4899', '#06b6d4'],
           showlegend=False),
    row=1, col=2
)

# Payment Method
payment_churn = df.groupby('PaymentMethod')['Churn_Binary'].mean() * 100
fig.add_trace(
    go.Bar(x=payment_churn.index, y=payment_churn.values,
           marker_color=['#f97316', '#84cc16', '#06b6d4', '#a855f7'],
           showlegend=False),
    row=2, col=1
)

# Senior Citizen
senior_churn = df.groupby('SeniorCitizen')['Churn_Binary'].mean() * 100
fig.add_trace(
    go.Bar(x=['No', 'Yes'], y=senior_churn.values,
           marker_color=['#10b981', '#ef4444'],
           showlegend=False),
    row=2, col=2
)

fig.update_yaxes(title_text="Churn Rate (%)", row=1, col=1)
fig.update_yaxes(title_text="Churn Rate (%)", row=2, col=1)

fig.update_layout(height=700, title_text="Churn Rate by Customer Segments")
fig.show()

print("\nSegment Insights:")
print(f"Month-to-month contracts: {contract_churn['Month-to-month']:.1f}% churn")
print(f"Two-year contracts: {contract_churn['Two year']:.1f}% churn")
print(f"Fiber optic users: {internet_churn['Fiber optic']:.1f}% churn")


Segment Insights:
Month-to-month contracts: 42.7% churn
Two-year contracts: 2.8% churn
Fiber optic users: 41.9% churn


## 5 FEATURE IMPORTANCE ANALYSIS

In [24]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Prepare data for modeling
df_model = df.copy()

# Select relevant features
features_to_encode = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
                      'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                      'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
                      'PaperlessBilling', 'PaymentMethod']

# Encode categorical variables
le = LabelEncoder()
for col in features_to_encode:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

# Select features
feature_cols = features_to_encode + ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
X = df_model[feature_cols]
y = df_model['Churn_Binary']

# Train Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_model.fit(X, y)

# Get feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False).head(15)

# Visualize
fig = go.Figure(go.Bar(
    x=feature_importance['importance'],
    y=feature_importance['feature'],
    orientation='h',
    marker=dict(
        color=feature_importance['importance'],
        colorscale='Viridis',
        showscale=True
    )
))

fig.update_layout(
    title='Top 15 Features Driving Customer Churn',
    xaxis_title='Importance Score',
    yaxis_title='Feature',
    height=600,
    yaxis={'categoryorder':'total ascending'}
)

fig.show()

print("\n Top 5 Churn Drivers:")
for idx, row in feature_importance.head(5).iterrows():
    print(f"{row['feature']}: {row['importance']:.3f}")


 Top 5 Churn Drivers:
TotalCharges: 0.161
tenure: 0.150
MonthlyCharges: 0.138
Contract: 0.129
OnlineSecurity: 0.076


## 6. REVENUE IMPACT ANALYSIS

In [25]:
# Calculate revenue metrics
churned_customers = df[df['Churn'] == 'Yes']
retained_customers = df[df['Churn'] == 'No']

total_revenue_lost = churned_customers['TotalCharges'].sum()
monthly_revenue_lost = churned_customers['MonthlyCharges'].sum()
avg_customer_value = churned_customers['TotalCharges'].mean()

# Create revenue comparison
revenue_comparison = pd.DataFrame({
    'Segment': ['Churned', 'Retained'],
    'Customers': [len(churned_customers), len(retained_customers)],
    'Avg Monthly Charges': [
        churned_customers['MonthlyCharges'].mean(),
        retained_customers['MonthlyCharges'].mean()
    ],
    'Total Revenue': [
        churned_customers['TotalCharges'].sum(),
        retained_customers['TotalCharges'].sum()
    ]
})

# Visualize
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Monthly Charges Comparison', 'Total Revenue by Segment'),
    specs=[[{"type": "box"}, {"type": "bar"}]]
)

# Box plot
fig.add_trace(
    go.Box(y=churned_customers['MonthlyCharges'], name='Churned',
           marker_color='#ef4444'),
    row=1, col=1
)
fig.add_trace(
    go.Box(y=retained_customers['MonthlyCharges'], name='Retained',
           marker_color='#10b981'),
    row=1, col=1
)

# Bar chart
fig.add_trace(
    go.Bar(x=revenue_comparison['Segment'],
           y=revenue_comparison['Total Revenue'],
           marker_color=['#ef4444', '#10b981'],
           text=revenue_comparison['Total Revenue'].apply(lambda x: f'${x/1e6:.1f}M'),
           textposition='outside'),
    row=1, col=2
)

fig.update_yaxes(title_text="Monthly Charges ($)", row=1, col=1)
fig.update_yaxes(title_text="Total Revenue ($)", row=1, col=2)

fig.update_layout(height=500, showlegend=True)
fig.show()

print("\nRevenue Impact:")
print(f"Total revenue from churned customers: ${total_revenue_lost:,.0f}")
print(f"Monthly recurring revenue lost: ${monthly_revenue_lost:,.0f}")
print(f"Average customer lifetime value (churned): ${avg_customer_value:,.0f}")
print(f"\nPotential annual revenue at risk: ${monthly_revenue_lost * 12:,.0f}")


Revenue Impact:
Total revenue from churned customers: $2,862,927
Monthly recurring revenue lost: $139,131
Average customer lifetime value (churned): $1,532

Potential annual revenue at risk: $1,669,570


## 7. COHORT ANALYSIS

In [26]:
# Create tenure cohorts and calculate retention
cohort_size = 6  # months
df['Cohort'] = (df['tenure'] // cohort_size) * cohort_size
df['Cohort_Label'] = df['Cohort'].astype(str) + '-' + (df['Cohort'] + cohort_size).astype(str) + ' months'

# Calculate cohort statistics
cohort_stats = df.groupby('Cohort_Label').agg({
    'customerID': 'count',
    'Churn_Binary': ['sum', 'mean'],
    'MonthlyCharges': 'mean',
    'TotalCharges': 'mean'
}).round(2)

cohort_stats.columns = ['Total Customers', 'Churned', 'Churn Rate', 'Avg Monthly Charge', 'Avg Total Charge']
cohort_stats['Retention Rate'] = (1 - cohort_stats['Churn Rate']) * 100
cohort_stats['Churn Rate'] = cohort_stats['Churn Rate'] * 100
cohort_stats = cohort_stats.reset_index()

# Sort by cohort
cohort_stats['sort_key'] = cohort_stats['Cohort_Label'].str.split('-').str[0].astype(int)
cohort_stats = cohort_stats.sort_values('sort_key').drop('sort_key', axis=1)

# Visualize retention curve
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=cohort_stats['Cohort_Label'],
    y=cohort_stats['Retention Rate'],
    mode='lines+markers',
    name='Retention Rate',
    line=dict(color='#10b981', width=4),
    marker=dict(size=10),
    fill='tozeroy',
    fillcolor='rgba(16, 185, 129, 0.2)'
))

fig.add_trace(go.Bar(
    x=cohort_stats['Cohort_Label'],
    y=cohort_stats['Total Customers'],
    name='Customer Count',
    yaxis='y2',
    marker_color='#3b82f6',
    opacity=0.6
))

fig.update_layout(
    title='Cohort Retention Analysis',
    xaxis_title='Tenure Cohort',
    yaxis_title='Retention Rate (%)',
    yaxis2=dict(
        title='Number of Customers',
        overlaying='y',
        side='right'
    ),
    height=500,
    hovermode='x unified'
)

fig.show()

print("\nCohort Statistics:")
print(cohort_stats[['Cohort_Label', 'Total Customers', 'Churn Rate', 'Retention Rate']].to_string(index=False))


Cohort Statistics:
Cohort_Label  Total Customers  Churn Rate  Retention Rate
  0-6 months             1360        55.0            45.0
 6-12 months              698        37.0            63.0
12-18 months              568        34.0            66.0
18-24 months              479        25.0            75.0
24-30 months              453        22.0            78.0
30-36 months              423        22.0            78.0
36-42 months              364        22.0            78.0
42-48 months              384        17.0            83.0
48-54 months              416        15.0            85.0
54-60 months              404        15.0            85.0
60-66 months              450         8.0            92.0
66-72 months              671         8.0            92.0
72-78 months              362         2.0            98.0


## 8. PREDICTIVE MODEL EVALUATION

In [27]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_model.fit(X_train, y_train)

# Predictions
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

# Visualize
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Confusion Matrix', 'ROC Curve'),
    specs=[[{"type": "heatmap"}, {"type": "scatter"}]]
)

# Confusion Matrix
fig.add_trace(
    go.Heatmap(
        z=cm,
        x=['Predicted No Churn', 'Predicted Churn'],
        y=['Actual No Churn', 'Actual Churn'],
        colorscale='Blues',
        text=cm,
        texttemplate='%{text}',
        textfont={"size": 16}
    ),
    row=1, col=1
)

# ROC Curve
fig.add_trace(
    go.Scatter(
        x=fpr, y=tpr,
        mode='lines',
        name=f'ROC (AUC = {roc_auc:.3f})',
        line=dict(color='#8b5cf6', width=3)
    ),
    row=1, col=2
)

fig.add_trace(
    go.Scatter(
        x=[0, 1], y=[0, 1],
        mode='lines',
        name='Random',
        line=dict(color='gray', dash='dash')
    ),
    row=1, col=2
)

fig.update_xaxes(title_text="False Positive Rate", row=1, col=2)
fig.update_yaxes(title_text="True Positive Rate", row=1, col=2)

fig.update_layout(height=500, showlegend=True)
fig.show()

# Print classification report
print("\nModel Performance:")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))
print(f"\nROC AUC Score: {roc_auc:.3f}")


Model Performance:
              precision    recall  f1-score   support

    No Churn       0.83      0.90      0.87      1549
       Churn       0.65      0.50      0.57       561

    accuracy                           0.80      2110
   macro avg       0.74      0.70      0.72      2110
weighted avg       0.79      0.80      0.79      2110


ROC AUC Score: 0.833


9/KEY INSIGHTS & RECOMMENDATIONS

In [28]:

print("KEY FINDINGS & ACTIONABLE RECOMMENDATIONS")


print("\n1.TENURE IMPACT")
print(f"   - First year churn rate: {churn_by_tenure.iloc[0]['Churn Rate']:.1f}%")
print(f"   - After 5 years: {churn_by_tenure.iloc[-1]['Churn Rate']:.1f}%")
print("   ➜ Focus on onboarding and early customer engagement")

print("\n2.CONTRACT TYPE")
print(f"   - Month-to-month: {contract_churn['Month-to-month']:.1f}% churn")
print(f"   - Two year: {contract_churn['Two year']:.1f}% churn")
print("   ➜ Incentivize longer contract commitments")

print("\n3 REVENUE AT RISK")
print(f"   - Annual revenue at risk: ${monthly_revenue_lost * 12:,.0f}")
print(f"   - Average churned customer value: ${avg_customer_value:,.0f}")
print("   ➜ Implement proactive retention programs")

print("\n4.HIGH-RISK SEGMENTS")
print("   - Fiber optic customers with month-to-month contracts")
print("   - Electronic check payment users")
print("   - Customers without tech support")
print("   ➜ Target these segments with retention campaigns")

print("\n5. MODEL INSIGHTS")
print(f"   - Model can predict churn with {roc_auc:.1%} accuracy")
print(f"   - Top drivers: {', '.join(feature_importance.head(3)['feature'].tolist())}")
print("   ➜ Use predictive model for early intervention")

print("Analysis Complete!")


KEY FINDINGS & ACTIONABLE RECOMMENDATIONS

1.TENURE IMPACT
   - First year churn rate: 47.7%
   - After 5 years: 6.6%
   ➜ Focus on onboarding and early customer engagement

2.CONTRACT TYPE
   - Month-to-month: 42.7% churn
   - Two year: 2.8% churn
   ➜ Incentivize longer contract commitments

3 REVENUE AT RISK
   - Annual revenue at risk: $1,669,570
   - Average churned customer value: $1,532
   ➜ Implement proactive retention programs

4.HIGH-RISK SEGMENTS
   - Fiber optic customers with month-to-month contracts
   - Electronic check payment users
   - Customers without tech support
   ➜ Target these segments with retention campaigns

5. MODEL INSIGHTS
   - Model can predict churn with 83.3% accuracy
   - Top drivers: TotalCharges, tenure, MonthlyCharges
   ➜ Use predictive model for early intervention
Analysis Complete!
